In [1]:
# Imports
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr
from xarray.coding.cftimeindex import CFTimeIndex
import fsspec

import warnings
from zarr.errors import ZarrUserWarning

import dask
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

c:\Users\Usuario\Documents\anaconda3\envs\env_cmip6_download2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

# Núcleos lógicos (threads com hyperthreading)
print("Núcleos lógicos (os.cpu_count()):", os.cpu_count())
n_workers = os.cpu_count() - 2  # ou 4, ou os.cpu_count()
print(f"🔧 Executando com {n_workers} workers (de {os.cpu_count()} disponíveis)")

Núcleos lógicos (os.cpu_count()): 8
🔧 Executando com 6 workers (de 8 disponíveis)


In [3]:
# Paths
# Loading path
path_aws = Path(r"C:\Users\Usuario\Documents\Python_Scripts\projeto-cnpq\cmip6-download-versao2\catalog\pangeo-cmip6_aws.csv")
path_google = Path(r"C:\Users\Usuario\Documents\Python_Scripts\projeto-cnpq\cmip6-download-versao2\catalog\pangeo-cmip6_google.csv")
path_download = Path(r"C:\Users\Usuario\Documents\Python_Scripts\projeto-cnpq\cmip6-download-versao2\downloads")

In [4]:
# Normalizing columns
def normalizar_campos(df):
    for col in ["source_id", "experiment_id", "member_id", "table_id", "variable_id", "version"]:
        df[col] = df[col].astype(str).str.strip()
    return df


# Verifying Keys
def criar_chave(df):
    return (
        df["source_id"] + "_" +
        df["experiment_id"] + "_" +
        df["member_id"] + "_" +
        df["table_id"] + "_" +
        df["variable_id"] + "_" +
        df["version"]
    )


def preprocess_time(ds, calendar="proleptic_gregorian"):
    # Ordena
    if not ds.indexes["time"].is_monotonic_increasing:
        ds = ds.sortby("time")

    # Converte calendário quando for CFTimeIndex
    if isinstance(ds.indexes["time"], CFTimeIndex):
        ds = ds.convert_calendar(calendar, align_on="year")
        # ---- REMOVER DUPLICATAS ----
        idx = ~pd.Index(ds.indexes["time"]).duplicated()
        if (~idx).any():
            ds = ds.isel(time=idx)

    else:
        # Aqui removemos a linha problemática:
        # ds["time"] = ds.indexes["time"].to_datetimeindex()

        # ---- REMOVER DUPLICATAS ----
        idx = ~ds.indexes["time"].duplicated()
        if (~idx).any():
            ds = ds.isel(time=idx)

    return ds


def adjust_coordinates_flex(ds):
    """
    Ajusta coordenadas de longitude para o intervalo [-180, 180] e ordena lat/lon se forem 1D.
    Suporta tanto coordenadas 1D (grade regular) quanto 2D (curvilinear).
    """
    import numpy as np

    lon_name = None
    lat_name = None

    # Detectar nomes das coordenadas
    for c in ds.coords:
        if c in ("lon", "longitude"): lon_name = c
        if c in ("lat", "latitude"):  lat_name = c

    # Ajustar longitude: [-180, 180]
    if lon_name is not None:
        lon = ds[lon_name]
        if lon.ndim == 1:
            # Grade regular
            lon_new = ((lon + 180) % 360) - 180
            ds = ds.assign_coords({lon_name: lon_new})
            ds = ds.sortby(lon_name)
        elif lon.ndim == 2:
            # Grade curvilinear
            lon_new = ((lon + 180) % 360) - 180
            ds = ds.assign_coords({lon_name: (lon.dims, lon_new)})
            # ❗ NÃO ordenamos 2D

    # Ordenar latitude se for 1D
    if lat_name is not None:
        lat = ds[lat_name]
        if lat.ndim == 1:
            ds = ds.sortby(lat_name)
        elif lat.ndim == 2:
            # Grade curvilinear → nada a fazer
            pass

    return ds


def crop_to_brazil(ds):
    """
    Recorta o dataset para os limites do Brasil, usando nomes dinâmicos de lat/lon.
    Só aplica o corte se as coordenadas forem 1D.
    Retorna sem recorte se a variável principal for 'tos'.
    """
    # Verifica se a variável principal é 'tos'
    if 'tos' in ds.data_vars:
        return ds  # Não recorta

    lat_name = None
    lon_name = None

    # Detectar nomes padrão
    for name in ds.coords:
        if name.lower() in ("lat", "latitude"):
            lat_name = name
        elif name.lower() in ("lon", "longitude"):
            lon_name = name

    # Se não encontrar lat/lon ou se forem 2D → retorna original
    if (
        lat_name is None or lon_name is None or
        ds[lat_name].ndim != 1 or ds[lon_name].ndim != 1
    ):
        return ds  # sem recorte

    # Recorte para os limites ampliados do Brasil
    return ds.sel(
        **{
            lat_name: slice(-70, 20),
            lon_name: slice(-120, -30)
        }
    )



def abrir_zarr_cmip6(row, consolidated=True):
    """
    Abre um dataset Zarr CMIP6 a partir de uma linha do catálogo.

    Retorna:
    - ds: xarray.Dataset (lazy, com Dask)
    - nome_arquivo: nome formatado com metadados
    """
    zarr_url = row["zstore"]
    try:
        mapper = fsspec.get_mapper(zarr_url, anon=True)
        ds = xr.open_zarr(mapper, consolidated=consolidated, chunks={})  # lazy
        nome_arquivo = f"{row['source_id']}_{row['experiment_id']}_{row['member_id']}_{row['table_id']}_{row['variable_id']}_{row['version']}.zarr"
        return ds, nome_arquivo
    except Exception as e:
        print(f"Erro ao abrir {zarr_url} -> {e}")
        return None, None


def salvar_dataset_em_blocos(ds, path_out, chunk_size=30):
    """
    Salva um xarray.Dataset em blocos ao longo da dimensão 'time',
    quebrando referências de compressão e evitando carregar tudo de uma vez.
    """

    # Verificação de segurança
    if "time" not in ds.dims:
        print(f"Aviso: dataset {path_out.name} não possui dimensão 'time'. Pulando.")
        return

    time_len = ds.sizes["time"]

    for i in range(0, time_len, chunk_size):
        bloco = ds.isel(time=slice(i, i + chunk_size)).load()

        ds_clean = xr.Dataset(
            data_vars={var: (bloco[var].dims, bloco[var].values) for var in bloco.data_vars},
            coords={coord: (bloco[coord].dims, bloco[coord].values) for coord in bloco.coords},
            attrs=bloco.attrs
        )

        mode = "w" if i == 0 else "a"

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ZarrUserWarning)
            if i == 0:
                ds_clean.to_zarr(path_out, mode=mode)
            else:
                ds_clean.to_zarr(path_out, mode=mode, append_dim="time")

        print(f"✔️ Bloco {i}–{min(i + chunk_size, time_len)} salvo em {path_out}")


def processar_linha(row):
    log_entry = {}
    ds, nome_arquivo = abrir_zarr_cmip6(row)
    path_out = path_download / nome_arquivo

    if path_out.exists():
        print(f"⚠️ Arquivo {path_out.name} já existe. Pulando.")
        return {"arquivo": nome_arquivo, "status": "Ignorado", "mensagem": "Arquivo já existe"}

    try:
        if ds is None:
            log_entry = {
                "arquivo": nome_arquivo or f"{row.name}",
                "status": "Erro",
                "mensagem": "Falha ao abrir Zarr"
            }

        elif "time" not in ds.dims:
            log_entry = {
                "arquivo": nome_arquivo,
                "status": "Sem dimensão time",
                "mensagem": ""
            }
            print(f"⚠️ Dataset {nome_arquivo} não possui dimensão 'time'. Pulando.")

        else:
            ds = preprocess_time(ds)
            ds = adjust_coordinates_flex(ds)
            ds = crop_to_brazil(ds)

            salvar_dataset_em_blocos(ds, path_out, chunk_size=5760)

            log_entry = {
                "arquivo": nome_arquivo,
                "status": "Sucesso",
                "mensagem": ""
            }

    except Exception as e:
        log_entry = {
            "arquivo": nome_arquivo or f"{row.name}",
            "status": "Erro",
            "mensagem": str(e)
        }

    return log_entry

In [5]:
# Opening CSV files
df_aws = pd.read_csv(path_aws)
df_google = pd.read_csv(path_google)

# Filtering
df_aws = df_aws.query('activity_id == "CMIP" and grid_label == "gn"')
df_google = df_google.query('activity_id == "CMIP" and grid_label == "gn"')

# Reset index
df_aws.reset_index(inplace=True)
df_google.reset_index(inplace=True)

# Drop dcpp_init_year
df_aws = df_aws.drop(columns=['dcpp_init_year', 'index', 'activity_id'], axis=1)
df_google = df_google.drop(columns=['dcpp_init_year', 'index', 'activity_id'], axis=1)

# Sorting data
ordenar_por = [
               "institution_id",  # Instituição responsável
               "source_id",       # Nome do modelo
               "experiment_id",   # Experimento (ex: historical, ssp245)
               "variable_id",     # Variável (ex: tas, pr)
               "table_id",        # Frequência temporal (ex: day, 3hr)
               "member_id",       # Membro do ensemble
               "version"          # Versão do dataset
               ]

df_aws = df_aws.sort_values(by=ordenar_por).reset_index(drop=True)
df_google = df_google.sort_values(by=ordenar_por).reset_index(drop=True)

In [6]:
# data
source_id_list = ['MIROC6', 'CMCC-ESM2', 'ACCESS-CM2', 'BCC-CSM2-MR', 'INM-CM5-0', 'EC-Earth3-Veg']
experiment_id_list = ['historical', 'ssp245', 'ssp585']
table_id_list = ['day', '3hr']
variable_id_list = ['tas', 'tasmax', 'huss', 'pr',
                    'wa', 'va', 'zg', 'wap',
                    'tos']

In [7]:
df_aws = df_aws[
                df_aws["source_id"].isin(source_id_list) &
                df_aws["experiment_id"].isin(experiment_id_list) &
                df_aws["table_id"].isin(table_id_list) &
                df_aws["variable_id"].isin(variable_id_list)
                ].copy()

df_aws.reset_index(drop=True, inplace=True)
df_aws.to_csv(path_download / Path("catalog_cnpq_aws.csv"), index=False)

df_google = df_google[
                      df_google["source_id"].isin(source_id_list) &
                      df_google["experiment_id"].isin(experiment_id_list) &
                      df_google["table_id"].isin(table_id_list) &
                      df_google["variable_id"].isin(variable_id_list)
                      ].copy()

df_google.reset_index(drop=True, inplace=True)
df_google.to_csv(path_download / Path("catalog_cnpq_google.csv"), index=False)


In [8]:
df_aws = normalizar_campos(df_aws)
df_google = normalizar_campos(df_google)

df_aws["chave"] = criar_chave(df_aws)
df_google["chave"] = criar_chave(df_google)

# 4. Remover duplicados (opcional)
df_aws.drop_duplicates(subset="chave", inplace=True)
df_google.drop_duplicates(subset="chave", inplace=True)

# 5. Diferença
chaves_aws = set(df_aws["chave"])
df_diff1 = df_google[~df_google["chave"].isin(chaves_aws)].copy()
df_diff1.drop(columns="chave", inplace=True)

# Debbuging
chaves_google = set(df_google['chave'])
df_diff2 = df_aws[~df_aws["chave"].isin(chaves_google)].copy()
df_diff2.drop(columns="chave", inplace=True)

# 6. Salvar resultado
df_diff1.to_csv(path_download / "catalog_diff_google_vs_aws.csv", index=False)
print("✅ Catálogo diferencial salvo com sucesso!")
df_diff2.to_csv(path_download / "catalog_diff_aws_vs_google.csv", index=False)
print("✅ Catálogo diferencial salvo com sucesso!")


✅ Catálogo diferencial salvo com sucesso!
✅ Catálogo diferencial salvo com sucesso!


In [ ]:
# Rodar paralelizado com ThreadPoolExecutor
df_selected = df_aws.copy()
log = []

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = {
        executor.submit(processar_linha, row): idx
        for idx, row in df_selected.tail(2).iterrows()  # Tirar o tail quando for rodar pra todo mundo
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Processando arquivos"):
        result = future.result()
        log.append(result)

# Salvar o log final
df_log = pd.DataFrame(log)
df_log.to_csv(path_download / "log_resultados_teste.csv", index=False)
print(f"\n📄 Log salvo em: {path_download / 'log_resultados_teste.csv'}")

In [4]:
path = r'C:\Users\Usuario\Documents\Python_Scripts\projeto-cnpq\cmip6-download-versao2\downloads\MIROC6_historical_r8i1p1f1_day_zg_20191016.zarr'
ds = xr.open_zarr(path)
ds

<xarray.Dataset> Size: 8GB
Dimensions:    (time: 60265, plev: 8, lat: 64, lon: 64, bnds: 2)
Coordinates:
  * lat        (lat) float64 512B -69.34 -67.94 -66.54 ... 16.11 17.51 18.91
    lat_bnds   (lat, bnds) float64 1kB dask.array<chunksize=(64, 2), meta=np.ndarray>
  * lon        (lon) float64 512B -119.5 -118.1 -116.7 ... -33.75 -32.34 -30.94
    lon_bnds   (lon, bnds) float64 1kB dask.array<chunksize=(64, 2), meta=np.ndarray>
  * plev       (plev) float64 64B 1e+05 8.5e+04 7e+04 ... 1e+04 5e+03 1e+03
  * time       (time) datetime64[ns] 482kB 1850-01-01T12:00:00 ... 2014-12-31...
    time_bnds  (time, bnds) datetime64[ns] 964kB dask.array<chunksize=(5760, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    zg         (time, plev, lat, lon) float32 8GB dask.array<chunksize=(720, 2, 16, 16), meta=np.ndarray>
Attributes: (12/45)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  76701.0
    cmor_version:           3.4.0
    ...                     ...
    table_id:               day
    table_info:             Creation Date:(22 July 2019) MD5:b4cefb4b6dbb146f...
    title:                  MIROC6 output prepared for CMIP6
    tracking_id:            hdl:21.14100/ccd08b42-2890-4397-91ee-c3f2706e18ef...
    variable_id:            zg
    variant_label:          r8i1p1f1